In [1]:
!pip install langchain langchain_community wikipedia tiktoken langchain-openai langchain-neo4j

In [2]:
from google.colab import userdata

In [17]:
# from langchain_neo4j import Neo4jGraph
from langchain_community.graphs import Neo4jGraph

# Neo4j connection setup
url = 'neo4j+s://b72f8b23.databases.neo4j.io'
username = userdata.get('USERNAME')
password = userdata.get('PASSWORD')
graph = Neo4jGraph(
    url=url,
    username=username,
    password=password
)

<ipython-input-17-19f173f8ed43>:8: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


In [18]:
# Use Pydantic V2
from pydantic import BaseModel, Field
from typing import List, Optional

class PropertyModel(BaseModel):
    key: str = Field(..., description="key")
    value: str = Field(..., description="value")

class NodeModel(BaseModel):
    id: str = Field(..., description="Node identifier")
    type: str = Field(..., description="Node type")
    properties: Optional[List[PropertyModel]] = Field(None, description="List of node properties")

class RelationshipModel(BaseModel):
    source: NodeModel = Field(..., description="Source node")
    target: NodeModel = Field(..., description="Target node")
    type: str = Field(..., description="Relationship type")
    properties: Optional[List[PropertyModel]] = Field(None, description="List of relationship properties")

class KnowledgeGraphModel(BaseModel):
    nodes: List[NodeModel] = Field(..., description="List of nodes in the knowledge graph")
    rels: List[RelationshipModel] = Field(..., description="List of relationships in the knowledge graph")

In [19]:
from langchain.graphs.graph_document import (
    Node as BaseNode,
    Relationship as BaseRelationship,
    GraphDocument,
)

def format_property_key(s: str) -> str:
    words = s.split()
    if not words:
        return s
    first_word = words[0].lower()
    capitalized_words = [word.capitalize() for word in words[1:]]
    return "".join([first_word] + capitalized_words)

def props_to_dict(props) -> dict:
    """Convert properties to a dictionary."""
    properties = {}
    if not props:
        return properties
    for p in props:
        properties[format_property_key(p.key)] = p.value
    return properties

def map_to_base_node(node: NodeModel) -> BaseNode:
    properties = props_to_dict(node.properties) if node.properties else {}
    properties["name"] = node.id.title()
    return BaseNode(
        id=node.id.title(),
        type=node.type.capitalize(),
        properties=properties
    )

def map_to_base_relationship(rel: RelationshipModel) -> BaseRelationship:
    source = map_to_base_node(rel.source)
    target = map_to_base_node(rel.target)
    properties = props_to_dict(rel.properties) if rel.properties else {}
    return BaseRelationship(
        source=source,
        target=target,
        type=rel.type,
        properties=properties
    )


In [20]:
import os
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

# Set OpenAI API key
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY').strip()
# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # Replace with your actual key
# llm = ChatOpenAI(model="gpt-3.5-turbo-16k", temperature=0)
llm = ChatOpenAI(model="gpt-4o-2024-08-06", temperature=0, api_key=OPENAI_API_KEY)

def get_extraction_chain(
    allowed_nodes: Optional[List[str]] = None,
    allowed_rels: Optional[List[str]] = None
):
    prompt = ChatPromptTemplate.from_messages(
        [(
          "system",
          f"""# Knowledge Graph Instructions for GPT-4
          ## 1. Overview
          You are a top-tier algorithm designed for extracting information in structured formats to build a knowledge graph.
          - **Nodes** represent entities and concepts. They're akin to Wikipedia nodes.
          - The aim is to achieve simplicity and clarity in the knowledge graph, making it accessible for a vast audience.
          ## 2. Labeling Nodes
          - **Consistency**: Ensure you use basic or elementary types for node labels.
            - For example, when you identify an entity representing a person, always label it as **"person"**. Avoid using more specific terms like "mathematician" or "scientist".
          - **Node IDs**: Never utilize integers as node IDs. Node IDs should be names or human-readable identifiers found in the text.
          {'- **Allowed Node Labels:** ' + ", ".join(allowed_nodes) if allowed_nodes else ""}
          {'- **Allowed Relationship Types**: ' + ", ".join(allowed_rels) if allowed_rels else ""}
          ## 3. Handling Numerical Data and Dates
          - Numerical data, like age or other related information, should be incorporated as attributes or properties of the respective nodes.
          - **No Separate Nodes for Dates/Numbers**: Do not create separate nodes for dates or numerical values. Always attach them as attributes or properties of nodes.
          - **Property Format**: Properties must be in a key-value format.
          - **Quotation Marks**: Never use escaped single or double quotes within property values.
          - **Naming Convention**: Use camelCase for property keys, e.g., `birthDate`.
          ## 4. Coreference Resolution
          - **Maintain Entity Consistency**: When extracting entities, it's vital to ensure consistency.
          If an entity, such as "John Doe", is mentioned multiple times in the text but is referred to by different names or pronouns (e.g., "Joe", "he"),
          always use the most complete identifier for that entity throughout the knowledge graph. In this example, use "John Doe" as the entity ID.
          ## 5. Strict Compliance
          Adhere to the rules strictly. Non-compliance will result in termination.
          """),
            ("human", "Use the given format to extract information from the following input: {input}"),
            ("human", "Tip: Make sure to answer in the correct format"),
        ])
    # Use the V2-compatible structured output method
    structured_llm = llm.with_structured_output(KnowledgeGraphModel)

    # Combine prompt and structured_llm into a chain
    chain = prompt | structured_llm
    return chain


In [21]:
from langchain.schema import Document

def extract_and_store_graph(
    document: Document,
    nodes: Optional[List[str]] = None,
    rels: Optional[List[str]] = None
) -> None:
    extract_chain = get_extraction_chain(nodes, rels)
    # Use invoke instead of run for the new method
    data = extract_chain.invoke({"input": document.page_content})
    graph_document = GraphDocument(
        nodes=[map_to_base_node(node) for node in data.nodes],
        relationships=[map_to_base_relationship(rel) for rel in data.rels],
        source=document
    )
    graph.add_graph_documents([graph_document])


In [22]:
from langchain.document_loaders import WikipediaLoader
from langchain.text_splitter import TokenTextSplitter

# Load and split documents
raw_documents = WikipediaLoader(query="Walt Disney").load()
text_splitter = TokenTextSplitter(chunk_size=2048, chunk_overlap=24)
documents = text_splitter.split_documents(raw_documents[:3])


In [24]:
from tqdm import tqdm

# Process documents
for i, d in tqdm(enumerate(documents), total=len(documents)):
    extract_and_store_graph(d)

100%|██████████| 3/3 [00:40<00:00, 13.50s/it]


In [25]:
# Delete the graph
graph.query("MATCH (n) DETACH DELETE n")

[]

In [26]:
allowed_nodes = ["Person", "Company", "Location", "Event", "Movie", "Service", "Award"]

for i, d in tqdm(enumerate(documents), total=len(documents)):
    extract_and_store_graph(d, allowed_nodes)

100%|██████████| 3/3 [00:35<00:00, 11.76s/it]


In [28]:
from langchain.chains import GraphCypherQAChain

graph.refresh_schema()

cypher_chain = GraphCypherQAChain.from_llm(
    graph=graph,
    cypher_llm=ChatOpenAI(temperature=0, model="gpt-4", api_key=OPENAI_API_KEY),
    qa_llm=ChatOpenAI(temperature=0, model="gpt-3.5-turbo", api_key=OPENAI_API_KEY),
    validate_cypher=True,
    verbose=True,
    allow_dangerous_requests=True

)

In [31]:
cypher_chain.invoke("When was Walter Disney born?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: "Walter Disney"}) RETURN p.birthdate
Full Context:
[]

> Finished chain.


{'query': 'When was Walter Disney born?', 'result': "I don't know the answer."}

In [32]:
cypher_chain.invoke("When was Walter Elias Disney born?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: "Walter Elias Disney"}) RETURN p.birthdate
Full Context:
[{'p.birthdate': '1901-12-05'}]

> Finished chain.


{'query': 'When was Walter Elias Disney born?',
 'result': 'Walter Elias Disney was born on December 5, 1901.'}